# 03 — Feature Engineering and Selection

**Goal:** turn cleaned transactions into time-aware customer prediction examples and build `return_30d` without future-data leakage.

The basic unit becomes **customer × prediction date**, not individual transaction.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
# When notebooks are launched from notebooks/, move to repository root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "raw" / "online_retail_II.xlsx"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "figures"
MODELS = PROJECT_ROOT / "models"

DATA_PROCESSED.mkdir(exist_ok=True, parents=True)
FIGURES.mkdir(exist_ok=True, parents=True)
MODELS.mkdir(exist_ok=True, parents=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_selection import SelectKBest, mutual_info_classif, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

purchase_path = DATA_PROCESSED / "purchase_transactions.csv"
df = pd.read_csv(purchase_path, parse_dates=["InvoiceDate"])

# Normalize identifiers after CSV load.
df["Customer ID"] = pd.to_numeric(df["Customer ID"], errors="coerce")
df["TotalPrice"] = pd.to_numeric(df["TotalPrice"], errors="coerce")

df = df.dropna(subset=["Customer ID", "InvoiceDate"]).copy()
df["Customer ID"] = df["Customer ID"].astype(int)

print(df.shape)
print(df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())

## 1. Choose monthly prediction dates

In [ ]:
# Monthly snapshots reduce the number of overlapping targets while using the full two-year history.
dates = pd.date_range(
    start=df["InvoiceDate"].min().normalize() + pd.offsets.MonthBegin(2),
    end=df["InvoiceDate"].max().normalize() - pd.Timedelta(days=31),
    freq="MS"
)
print("Prediction dates:", len(dates))
display(pd.DataFrame({"prediction_date": dates}))

## 2. Build leakage-safe customer features

In [ ]:
def build_snapshot(transactions, prediction_date, lookback_days=180, horizon_days=30):
    past_start = prediction_date - pd.Timedelta(days=lookback_days)
    future_end = prediction_date + pd.Timedelta(days=horizon_days)

    past = transactions[
        (transactions["InvoiceDate"] < prediction_date) &
        (transactions["InvoiceDate"] >= past_start)
    ].copy()

    future = transactions[
        (transactions["InvoiceDate"] >= prediction_date) &
        (transactions["InvoiceDate"] < future_end)
    ][["Customer ID", "InvoiceDate"]].copy()

    if past.empty:
        return pd.DataFrame()

    grouped = past.groupby("Customer ID")

    feat = grouped.agg(
        RecencyDays=("InvoiceDate", lambda s: (prediction_date - s.max()).days),
        TransactionCount=("Invoice", "nunique"),
        PurchaseDays=("InvoiceDate", lambda s: s.dt.date.nunique()),
        TotalSpend=("TotalPrice", "sum"),
        AverageOrderValue=("TotalPrice", "mean"),
        TotalQuantity=("Quantity", "sum"),
        UniqueProducts=("StockCode", "nunique"),
        ActiveMonths=("InvoiceDate", lambda s: s.dt.to_period("M").nunique())
    ).reset_index()

    # Customer country: most recently observed country before prediction date.
    country = (
        past.sort_values("InvoiceDate")
            .drop_duplicates("Customer ID", keep="last")[["Customer ID", "Country"]]
    )
    feat = feat.merge(country, on="Customer ID", how="left")

    feat["AvgItemsPerTransaction"] = feat["TotalQuantity"] / feat["TransactionCount"].replace(0, np.nan)
    feat["PurchaseDaysPerMonth"] = feat["PurchaseDays"] / feat["ActiveMonths"].replace(0, np.nan)

    # Target: did this customer purchase in the next 30 days?
    buyers_next_30 = set(future["Customer ID"].dropna().astype(int).unique())
    feat["return_30d"] = feat["Customer ID"].isin(buyers_next_30).astype(int)
    feat["PredictionDate"] = prediction_date

    return feat

snapshots = []
for d in dates:
    snap = build_snapshot(df, d)
    if not snap.empty:
        snapshots.append(snap)

model_df = pd.concat(snapshots, ignore_index=True)

print("Customer-period dataset:", model_df.shape)
print(model_df["return_30d"].value_counts(normalize=True).rename("proportion"))
display(model_df.head())

## 3. Check target construction

In [ ]:
target_check = (
    model_df.groupby("PredictionDate")["return_30d"]
            .agg(["count", "mean", "sum"])
)
display(target_check)

## 4. Prepare numeric feature matrix for feature-selection experiments

In [ ]:
numeric_features = [
    "RecencyDays",
    "TransactionCount",
    "PurchaseDays",
    "TotalSpend",
    "AverageOrderValue",
    "TotalQuantity",
    "UniqueProducts",
    "ActiveMonths",
    "AvgItemsPerTransaction",
    "PurchaseDaysPerMonth"
]

X_num = model_df[numeric_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = model_df["return_30d"].astype(int)

# Time-based split before supervised selection.
train_cut = model_df["PredictionDate"].quantile(0.70)
val_cut = model_df["PredictionDate"].quantile(0.85)

train_mask = model_df["PredictionDate"] < train_cut
val_mask = (model_df["PredictionDate"] >= train_cut) & (model_df["PredictionDate"] < val_cut)
test_mask = model_df["PredictionDate"] >= val_cut

print("Train:", train_mask.sum(), "Validation:", val_mask.sum(), "Test:", test_mask.sum())

## 5. Feature-selection experiment on training data only

In [ ]:
selector = SelectKBest(score_func=f_classif, k="all")
selector.fit(X_num.loc[train_mask], y.loc[train_mask])

feature_scores = pd.DataFrame({
    "feature": numeric_features,
    "F_score": selector.scores_,
    "p_value": selector.pvalues_
}).sort_values("F_score", ascending=False)

display(feature_scores)

## 6. PCA / dimensionality-reduction experiment

In [ ]:
scaler_pca = StandardScaler()
X_train_scaled = scaler_pca.fit_transform(X_num.loc[train_mask])

pca_full = PCA().fit(X_train_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

pca_report = pd.DataFrame({
    "component": np.arange(1, len(cum_var)+1),
    "explained_variance_ratio": pca_full.explained_variance_ratio_,
    "cumulative_variance": cum_var
})
display(pca_report)

plt.figure(figsize=(8, 5))
plt.plot(pca_report["component"], pca_report["cumulative_variance"], marker="o")
plt.axhline(0.90, linestyle="--", linewidth=1)
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA Cumulative Explained Variance")
plt.tight_layout()
plt.savefig(FIGURES / "pca_explained_variance.png", dpi=160)
plt.show()

# Retain the smallest number of components explaining at least 90%.
pca_components_90 = int(np.argmax(cum_var >= 0.90) + 1)
print("Components explaining >= 90% variance:", pca_components_90)

## 7. Save the engineered modeling data

In [ ]:
model_path = DATA_PROCESSED / "customer_modeling_data.csv"
model_df.to_csv(model_path, index=False)

feature_score_path = DATA_PROCESSED / "feature_selection_scores.csv"
feature_scores.to_csv(feature_score_path, index=False)

print("Saved:", model_path)
print("Saved:", feature_score_path)

### Decision to record in the report

After comparing the selection and PCA results, document:
- the feature-selection method;
- the ranking/scores;
- the chosen threshold or K;
- the PCA component count and explained variance;
- whether PCA is retained in the final pipeline and why.

Do not choose these solely because they produce the best test result; the final decision must be justified and based on the training/validation process.